# QLoRA fine-tune: small open coder model for DataAgent text-to-SQL

Runs on a free Colab/Kaggle **T4 (16 GB)** in roughly 1-2 hours for a 1.5B model.

- Base: `Qwen2.5-Coder-1.5B-Instruct` (swap `BASE` for a 3B/7B model on a bigger GPU).
- Data: `training/build_dataset.py` — Spider **train** only, rendered with the exact
  prompt DataAgent sends at inference. Never dev (the builder refuses).
- Output: a LoRA adapter and a `q4_k_m` GGUF for Ollama, optionally pushed to the Hugging Face Hub.

Then evaluate locally with the full statistics treatment:
`ollama create dataagent-sql -f Modelfile` and
`python -m benchmarks.spider_eval --model ollama/dataagent-sql --subset 200`.

In [ ]:
!pip -q install unsloth
!git clone -q https://github.com/AHZ003/data-agent.git
%cd data-agent
!pip -q install -r requirements.txt
!bash benchmarks/spider_setup.sh
!python -m training.build_dataset

In [ ]:
BASE = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ = 4096
HF_REPO = None  # e.g. "your-name/dataagent-qwen2.5-coder-1.5b-sql" to push

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0,
)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "training/data/train.jsonl", "val": "training/data/val.jsonl"})

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}

ds = ds.map(to_text, batched=True)
# Drop the few prompts longer than the context rather than truncating the answer away.
ds = ds.filter(lambda ex: len(tokenizer(ex["text"]).input_ids) <= MAX_SEQ)
print(ds)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
        logging_steps=20, eval_strategy="steps", eval_steps=200,
        optim="adamw_8bit", seed=0, output_dir="training/outputs", report_to="none",
    ),
)
# Loss only on the SQL answer, not on the (long) prompt.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

In [ ]:
# Quick sanity check on held-out databases (exact-match only; run the real
# execution-accuracy eval locally through DataAgent's harness).
import json, re
FastLanguageModel.for_inference(model)
val = [json.loads(l) for l in open("training/data/val.jsonl")][:20]
for ex in val[:5]:
    msgs = ex["messages"][:1]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=256, do_sample=False)
    print("PRED:", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())
    print("GOLD:", ex["messages"][1]["content"], "\n")

In [ ]:
model.save_pretrained("training/outputs/adapter")
tokenizer.save_pretrained("training/outputs/adapter")
model.save_pretrained_gguf("training/outputs/gguf", tokenizer, quantization_method="q4_k_m")
if HF_REPO:
    from huggingface_hub import notebook_login
    notebook_login()
    model.push_to_hub(HF_REPO + "-lora")
    model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method="q4_k_m")

## Serve with Ollama (on your machine)

Download the `.gguf` from `training/outputs/gguf/`, then:

```
cat > Modelfile <<'MF'
FROM ./unsloth.Q4_K_M.gguf
PARAMETER temperature 0
PARAMETER num_ctx 4096
MF
ollama create dataagent-sql -f Modelfile
python -m benchmarks.spider_eval --model ollama/dataagent-sql --subset 200 --seed 0
```